In [2]:
import pandas as pd

df = pd.read_csv("enron_spam_data.csv")

print(df.head())

   Message ID                       Subject  \
0           0  christmas tree farm pictures   
1           1      vastar resources , inc .   
2           2  calpine daily gas nomination   
3           3                    re : issue   
4           4     meter 7268 nov allocation   

                                             Message Spam/Ham        Date  
0                                                NaN      ham  1999-12-10  
1  gary , production from the high island larger ...      ham  1999-12-13  
2             - calpine daily gas nomination 1 . doc      ham  1999-12-14  
3  fyi - see note below - already done .\nstella\...      ham  1999-12-14  
4  fyi .\n- - - - - - - - - - - - - - - - - - - -...      ham  1999-12-14  


In [3]:
print(df.columns)

Index(['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date'], dtype='object')


In [5]:
print(len(df))

33716


In [6]:
df.head(5)

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [7]:
# handle missing values
df = df.dropna(subset=['Message'])

In [8]:
# combine subject and message into a single text column
df['text'] = df['Subject'] + ' ' + df['Message']

C:\Users\kotha\AppData\Local\Temp\ipykernel_26544\2403322549.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['Subject'] + ' ' + df['Message']


In [10]:
# rename label column
df = df.rename(columns={'Spam/Ham': 'label'})

df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [11]:
# Keep only the relevant columns
df = df[['text', 'label']]

In [12]:
# verify the changes
print(df.head())

                                                text  label
1  vastar resources , inc . gary , production fro...      0
2  calpine daily gas nomination - calpine daily g...      0
3  re : issue fyi - see note below - already done...      0
4  meter 7268 nov allocation fyi .\n- - - - - - -...      0
5  mcmullen gas for 11 / 99 jackie ,\nsince the i...      0


In [13]:
print(df['label'].value_counts())

label
1    16852
0    16493
Name: count, dtype: int64


In [14]:
print(df.isnull().sum())

text     238
label      0
dtype: int64


In [15]:
df = df.dropna(subset=['text']) 

In [19]:
df = df[df['text'].str.strip() != ""]  # Remove rows where 'text' is empty or contains only whitespace

In [17]:
print(df.isnull().sum())
print(len(df))

text     0
label    0
dtype: int64
33107


In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # lowercase
    text = text.lower()
    
    # remove numbers
    text = re.sub(r'\d+', '', text) # remove digits
    
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text) # remove punctuation except for word characters and whitespace
    
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove leading/trailing whitespace and reduce multiple spaces to single
    
    # remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kotha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [21]:
df['clean_text'] = df['text'].apply(clean_text)

In [23]:
print("Before:\n", df['text'].iloc[0])
print("\nAfter:\n", df['clean_text'].iloc[0])

Before:
 vastar resources , inc . gary , production from the high island larger block a - 1 # 2 commenced on
saturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and
10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .
george x 3 - 6992
- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16
am - - - - - - - - - - - - - - - - - - - - - - - - - - -
daren j farmer
12 / 10 / 99 10 : 38 am
to : carlos j rodriguez / hou / ect @ ect
cc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect
subject : vastar resources , inc .
carlos ,
please call linda and get everything set up .
i ' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each
following day based on my conversations with bill fischer at bmar .
d .
- - - - - - - - - - - - - - - - - - - - - - forwarded by daren j farmer / hou / ect on 12 / 10 / 99 10 : 34
am - - - - - - - - - - - - - - - - - - -